# 07 — Results Comparison

Aggregates all experiments from `results.csv` and compares custom pipeline configurations against
the BERTopic baseline using the **three-tier evaluation framework**:

- **Tier 1** — Silhouette, Davies-Bouldin, Calinski-Harabasz (geometric separation)
- **Tier 2** — C_v topic coherence (lexical distinctiveness, independent of embedding geometry)
- **Tier 3** — Rating entropy (thematic vs. sentiment-dominated cluster detection)

See `06_clustering.ipynb` for a detailed explanation of why each tier matters.

**Prerequisite:** Run notebooks 03–06 first.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
sys.path.insert(0, '..')

from utils import load_config, load_results

cfg = load_config(sample_size=5_000)
df = load_results(cfg.results_csv)

print(f"Total experiments logged: {len(df)}")
print(f"Pipelines: {df['pipeline'].value_counts().to_dict()}")
has_coherence = df["coherence_cv"].notna().any()
has_entropy   = df["rating_entropy"].notna().any()
print(f"Tier 2 coherence data: {has_coherence}  |  Tier 3 entropy data: {has_entropy}")

## Sanity checks

In [ ]:
dup = df["run_id"].duplicated().sum()
assert dup == 0, f"{dup} duplicate run_ids — re-run log_result with same params?"
print(f"No duplicate run_ids ({len(df)} unique experiments)")

sil = df["silhouette"].dropna()
assert (sil >= -1).all() and (sil <= 1).all()
print(f"Silhouette range: [{sil.min():.3f}, {sil.max():.3f}]")

n_bt = (df["pipeline"] == "bertopic").sum()
print(f"BERTopic rows: {n_bt}")

print("\nMean metrics per pipeline:")
agg_cols = [c for c in ["n_clusters", "silhouette", "coherence_cv", "rating_entropy"] if c in df.columns]
print(df.groupby("pipeline")[agg_cols].mean().round(3))

## Summary table — top 15 by silhouette

Silhouette sorts the table, but check coherence and entropy columns before picking a winner.
High silhouette with low coherence often signals a sentiment-collapsed configuration.

In [ ]:
cols = ["pipeline", "embedding_model", "embedding_instruction",
        "reduction_method", "clustering_algo",
        "n_clusters", "noise_ratio", "silhouette",
        "coherence_cv", "rating_entropy", "runtime_s"]

top15 = (
    df[cols]
    .dropna(subset=["silhouette"])
    .sort_values("silhouette", ascending=False)
    .head(15)
    .reset_index(drop=True)
)
top15["embedding_model"] = top15["embedding_model"].str.split("/").str[-1]
top15

## Chart 1 — Top 10 by silhouette (with coherence overlay)

Bars = silhouette (Tier 1). Green dots = C_v coherence (Tier 2).
A bar with a low green dot despite high silhouette is a warning sign.

In [ ]:
plot_df = top15.head(10).copy()
plot_df["label"] = (
    plot_df["embedding_model"].str[:18] + "\n"
    + plot_df["embedding_instruction"].fillna("") + " / "
    + plot_df["clustering_algo"]
)
bar_colors = ["#888888" if p == "bertopic" else "steelblue" for p in plot_df["pipeline"]]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(len(plot_df)), plot_df["silhouette"], color=bar_colors, alpha=0.8)

if has_coherence:
    ax2 = ax.twinx()
    ax2.scatter(range(len(plot_df)), plot_df["coherence_cv"],
                color="seagreen", s=100, zorder=5, label="C_v coherence")
    ax2.set_ylabel("C_v coherence (Tier 2)", color="seagreen")
    ax2.tick_params(axis='y', labelcolor="seagreen")
    ax2.legend(loc="upper right")

ax.set_xticks(range(len(plot_df)))
ax.set_xticklabels(plot_df["label"], fontsize=8)
ax.set_ylabel("Silhouette (Tier 1)")
ax.set_title("Top 10 configurations — silhouette (bars) vs C_v coherence (dots)\n"
             "grey=BERTopic | blue=custom")
legend_handles = [
    mpatches.Patch(color="#888888", label="BERTopic"),
    mpatches.Patch(color="steelblue", label="Custom pipeline"),
]
ax.legend(handles=legend_handles)
plt.tight_layout()
plt.show()

## Chart 2 — Silhouette × coherence quadrant scatter (primary decision chart)

Each point is one experiment. The **top-right quadrant** contains the real winners:
configurations that are both geometrically tight (high silhouette) and lexically distinct
(high C_v coherence).

- **Top-right**: both metrics high → ideal topic clusters
- **Bottom-right**: high silhouette but low coherence → likely sentiment blobs, not topics
- **Top-left**: high coherence but low silhouette → distinct topics, imprecise geometry
- **Bottom-left**: poor on both → discard

In [ ]:
quad_df = df[df["silhouette"].notna() & df["coherence_cv"].notna()].copy()

if len(quad_df) == 0:
    print("No coherence data yet. Set COMPUTE_COHERENCE=True in notebooks 03/04/06 and re-run.")
else:
    fig, ax = plt.subplots(figsize=(11, 7))

    algo_colors = {
        "hdbscan":           "steelblue",
        "kmeans":            "coral",
        "agglomerative":     "mediumpurple",
        "bertopic_internal": "#888888",
    }
    pipeline_markers = {"custom": "o", "bertopic": "s"}

    for _, row in quad_df.iterrows():
        color  = algo_colors.get(row.get("clustering_algo", ""), "black")
        marker = pipeline_markers.get(row["pipeline"], "o")
        ax.scatter(row["silhouette"], row["coherence_cv"],
                   color=color, marker=marker, s=80, alpha=0.8)

    # Reference lines at medians to define the quadrants
    med_sil = quad_df["silhouette"].median()
    med_coh = quad_df["coherence_cv"].median()
    ax.axvline(med_sil, color="grey", linestyle="--", alpha=0.4,
               label=f"median sil={med_sil:.2f}")
    ax.axhline(med_coh, color="grey", linestyle=":", alpha=0.4,
               label=f"median coh={med_coh:.2f}")

    # Quadrant labels
    xlim = ax.get_xlim(); ylim = ax.get_ylim()
    ax.text(xlim[1] * 0.98, ylim[1] * 0.98, "REAL WINNERS",
            ha="right", va="top", fontsize=9, color="seagreen", alpha=0.7)
    ax.text(xlim[1] * 0.98, ylim[0] * 0.02, "Sentiment blobs?",
            ha="right", va="bottom", fontsize=8, color="tomato", alpha=0.7)

    for algo, color in algo_colors.items():
        ax.scatter([], [], color=color, label=algo, s=60)
    ax.scatter([], [], marker="o", color="grey", label="custom", s=60)
    ax.scatter([], [], marker="s", color="grey", label="bertopic", s=80)
    ax.legend(fontsize=8, ncol=2)

    ax.set_xlabel("Silhouette — geometric separation (Tier 1)")
    ax.set_ylabel("C_v coherence — lexical distinctiveness (Tier 2)")
    ax.set_title("Silhouette × C_v coherence — top-right = real winners")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## Chart 3 — Rating entropy by configuration (Tier 3)

Configurations above the green line produce thematic clusters (star ratings mixed → topic-driven).
Configurations below the red line produce sentiment blobs (one star value dominates each cluster).

In [ ]:
entr_df = df[df["rating_entropy"].notna()].copy()

if len(entr_df) == 0:
    print("No rating entropy data yet.")
else:
    entr_df["label"] = (
        entr_df["embedding_model"].str.split("/").str[-1].str[:14] + "\n"
        + entr_df["clustering_algo"]
    )
    entr_df_sorted = entr_df.sort_values("rating_entropy", ascending=False)

    bar_colors = [
        "seagreen" if e >= 0.85 else ("goldenrod" if e >= 0.60 else "tomato")
        for e in entr_df_sorted["rating_entropy"]
    ]

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(range(len(entr_df_sorted)), entr_df_sorted["rating_entropy"],
           color=bar_colors, alpha=0.8)
    ax.axhline(0.85, color="seagreen", linestyle="--", alpha=0.6,
               label="thematic threshold (>0.85)")
    ax.axhline(0.60, color="tomato", linestyle="--", alpha=0.6,
               label="sentiment blob threshold (<0.60)")
    ax.set_xticks(range(len(entr_df_sorted)))
    ax.set_xticklabels(entr_df_sorted["label"], fontsize=7, rotation=45, ha="right")
    ax.set_ylabel("Normalised star-rating entropy (Tier 3)")
    ax.set_title("Rating entropy — green=thematic | yellow=mixed | red=sentiment blob")
    ax.legend()
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.show()

## Chart 4 — Silhouette vs UMAP n_components (per embedding model)

In [ ]:
nc_df = df[
    (df["pipeline"] == "custom") &
    (df["reduction_method"] == "umap") &
    (df["clustering_algo"] == "hdbscan") &
    df["umap_n_components"].notna() &
    df["silhouette"].notna()
].copy()

if len(nc_df) == 0:
    print("No n_components sweep data yet — run 05_dimensionality_reduction.ipynb first.")
else:
    nc_df["model_short"] = nc_df["embedding_model"].str.split("/").str[-1]
    fig, ax = plt.subplots(figsize=(10, 5))
    for model, group in nc_df.groupby("model_short"):
        g = group.sort_values("umap_n_components")
        ax.plot(g["umap_n_components"], g["silhouette"], "o-", label=model)
    ax.set_xlabel("UMAP n_components")
    ax.set_ylabel("Silhouette")
    ax.set_title("Silhouette vs UMAP n_components — per embedding model")
    ax.legend(fontsize=8, bbox_to_anchor=(1, 1))
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## Chart 5 — Instruction effect (two heatmaps: silhouette and coherence)

In [ ]:
instr_models = df[
    df["embedding_model"].str.contains("instructor|e5.*instruct|Qwen3", case=False, na=False)
]["embedding_model"].unique()

instr_df = df[
    df["embedding_model"].isin(instr_models) & df["silhouette"].notna()
].copy()
instr_df["model_short"] = instr_df["embedding_model"].str.split("/").str[-1]

if len(instr_df) == 0:
    print("No instruction-tuned model results yet.")
else:
    n_subplots = 2 if has_coherence else 1
    fig, axes = plt.subplots(1, n_subplots,
                              figsize=(10 * n_subplots, max(3, len(instr_df["model_short"].unique()) * 0.8 + 1)))
    if n_subplots == 1:
        axes = [axes]

    pivot_sil = instr_df.pivot_table(
        index="model_short", columns="embedding_instruction",
        values="silhouette", aggfunc="max"
    )
    sns.heatmap(pivot_sil, annot=True, fmt=".3f", cmap="YlOrRd", ax=axes[0])
    axes[0].set_title("Silhouette by model × instruction (Tier 1)")

    if has_coherence:
        pivot_coh = instr_df.pivot_table(
            index="model_short", columns="embedding_instruction",
            values="coherence_cv", aggfunc="max"
        )
        sns.heatmap(pivot_coh, annot=True, fmt=".3f", cmap="YlGn", ax=axes[1])
        axes[1].set_title("C_v coherence by model × instruction (Tier 2)")

    plt.tight_layout()
    plt.show()

## Chart 6 — Speed vs quality (colour = coherence)

In [ ]:
sv_df = df[df["silhouette"].notna() & df["runtime_s"].notna()].copy()

fig, ax = plt.subplots(figsize=(11, 6))
pipeline_marker = {"custom": "o", "bertopic": "s"}

for _, row in sv_df.iterrows():
    marker = pipeline_marker.get(row["pipeline"], "o")
    if has_coherence and pd.notna(row.get("coherence_cv")):
        c = plt.cm.RdYlGn(float(row["coherence_cv"]))
    else:
        c = "steelblue" if row["pipeline"] == "custom" else "#888888"
    ax.scatter(row["runtime_s"], row["silhouette"],
               marker=marker, color=c, s=80, alpha=0.8)

# Annotate Pareto-optimal points (top silhouette per runtime bucket)
for _, row in sv_df.nlargest(5, "silhouette").iterrows():
    ax.annotate(
        row["embedding_model"].split("/")[-1][:16],
        (row["runtime_s"], row["silhouette"]),
        fontsize=7, xytext=(5, 5), textcoords="offset points"
    )

ax.set_xlabel("Total runtime (s)")
ax.set_ylabel("Silhouette")
ax.set_title("Speed vs silhouette (colour = C_v coherence, green=high, circles=custom, squares=BERTopic)")
legend_handles = [
    mpatches.Patch(color="#888888", label="BERTopic"),
    mpatches.Patch(color="steelblue", label="Custom pipeline"),
]
ax.legend(handles=legend_handles)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## BERTopic vs best custom pipeline (all three tiers)

In [ ]:
best_custom   = df[df["pipeline"] == "custom"]["silhouette"].max()
best_bertopic = df[df["pipeline"] == "bertopic"]["silhouette"].max()

print("── Tier 1: Silhouette ──")
print(f"  Best custom   : {best_custom}")
print(f"  Best BERTopic : {best_bertopic}")
print(f"  Delta         : {best_custom - best_bertopic:+.4f}")

if has_coherence:
    best_coh_c = df[df["pipeline"] == "custom"]["coherence_cv"].max()
    best_coh_b = df[df["pipeline"] == "bertopic"]["coherence_cv"].max()
    print("\n── Tier 2: C_v coherence ──")
    print(f"  Best custom   : {best_coh_c}")
    print(f"  Best BERTopic : {best_coh_b}")
    print(f"  Delta         : {best_coh_c - best_coh_b:+.4f}")

if has_entropy:
    best_entr_c = df[df["pipeline"] == "custom"]["rating_entropy"].max()
    best_entr_b = df[df["pipeline"] == "bertopic"]["rating_entropy"].max()
    print("\n── Tier 3: Rating entropy ──")
    print(f"  Best custom   : {best_entr_c}")
    print(f"  Best BERTopic : {best_entr_b}")
    print(f"  Delta         : {best_entr_c - best_entr_b:+.4f}")

print("\nBest custom config:")
print(df[df["pipeline"] == "custom"].nlargest(1, "silhouette")[
    ["embedding_model", "embedding_instruction", "reduction_method",
     "clustering_algo", "n_clusters", "silhouette", "coherence_cv", "runtime_s"]
].to_string(index=False))

## Findings summary

| Question | Sil winner (T1) | Coherence winner (T2) | Entropy winner (T3) |
|---|---|---|---|
| Best embedding model | _fill_ | _fill_ | _fill_ |
| Best instruction | _fill_ | _fill_ | _fill_ |
| Best UMAP n_components | _fill_ | _fill_ | _fill_ |
| Best clustering algorithm | _fill_ | _fill_ | _fill_ |
| Custom pipeline beats BERTopic? | _fill_ | _fill_ | _fill_ |

**Overall winner (all three tiers):** _fill in_ — this is the configuration to use in production.

**Interesting divergences** (high silhouette, low coherence or vice versa): _fill in_
These reveal where sentiment polarity is being confused with topical structure.

**Use these findings to configure the production NLP pipeline in the ReviewScope backend.**